# ZNO Data Analysis

To run this notebook, make sure you have installed `pandas`, `matplotlib`, and `numpy`.

To download datasets, you can run `download.sh` bash script or manually from zno.testportal.com.ua/opendata site. Make sure you have `7zz` installed to extract data from archives.

`bash download.sh 2016 2017 2018 2019 2020 2021`

In [11]:
import numpy as np
import pandas as pd
import re
import json
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import geopandas as gpd

from pathlib import Path
from matplotlib.widgets import Slider, RadioButtons
from ipywidgets import interact, IntRangeSlider, Dropdown


In [2]:
data_path = Path("./data")

In [3]:
years_dict = {}

In [4]:
for file in data_path.glob("*.csv"):
    filename = file.name
    year = re.search(r"20[0-9]{2}", filename).group()
    try:
        df = pd.read_csv(file, sep=";", low_memory=False, encoding="utf-8-sig")
        print(f"Loaded {year} successfully with utf-8-sig")
        
    except UnicodeDecodeError:
        df = pd.read_csv(file, sep=";", low_memory=False, encoding="cp1251")
        print(f"Loaded {year} successfully with cp1251")
            
    years_dict[year] = df

Loaded 2016 successfully with cp1251
Loaded 2017 successfully with utf-8-sig
Loaded 2018 successfully with utf-8-sig
Loaded 2019 successfully with cp1251
Loaded 2020 successfully with cp1251
Loaded 2021 successfully with utf-8-sig


In [5]:
years_dict.keys()

dict_keys(['2016', '2017', '2018', '2019', '2020', '2021'])

In [6]:
df = list(years_dict.values())[-1]

for df in years_dict.values():
    df.columns = df.columns.str.lower()
df.columns.to_list()

['outid',
 'birth',
 'sextypename',
 'regname',
 'areaname',
 'tername',
 'regtypename',
 'tertypename',
 'classprofilename',
 'classlangname',
 'eoname',
 'eotypename',
 'eoregname',
 'eoareaname',
 'eotername',
 'eoparent',
 'umltest',
 'umlteststatus',
 'umlball100',
 'umlball12',
 'umlball',
 'umladaptscale',
 'umlptname',
 'umlptregname',
 'umlptareaname',
 'umlpttername',
 'ukrtest',
 'ukrsubtest',
 'ukrteststatus',
 'ukrball100',
 'ukrball12',
 'ukrball',
 'ukradaptscale',
 'ukrptname',
 'ukrptregname',
 'ukrptareaname',
 'ukrpttername',
 'histtest',
 'histlang',
 'histteststatus',
 'histball100',
 'histball12',
 'histball',
 'histptname',
 'histptregname',
 'histptareaname',
 'histpttername',
 'mathtest',
 'mathlang',
 'mathteststatus',
 'mathball100',
 'mathball12',
 'mathdpalevel',
 'mathball',
 'mathptname',
 'mathptregname',
 'mathptareaname',
 'mathpttername',
 'mathsttest',
 'mathstlang',
 'mathstteststatus',
 'mathstball12',
 'mathstball',
 'mathstptname',
 'mathstptregn

In [7]:
df.head()

,outid,birth,sextypename,regname,areaname,tername,regtypename,tertypename,classprofilename,classlangname,...,spatest,spateststatus,spaball100,spaball12,spadpalevel,spaball,spaptname,spaptregname,spaptareaname,spapttername
0,8a2abef7-625a-4253-8c14-000fe8a856e5,2003,чоловіча,Дніпропетровська область,м.Кам'янське,Дніпровський район міста,Студент закладу вищої освіти,місто,Молодший спеціаліст,українська,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3e975d01-9cc9-4072-bbcd-005b9da75096,2003,чоловіча,Кіровоградська область,м.Кропивницький,Фортечний район міста,Випускник загальноосвітнього навчального закла...,місто,Технологічний,українська,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,58c926c3-8718-4845-918b-00155b917934,2003,чоловіча,Дніпропетровська область,м.Дніпро,Соборний район міста,Студент закладу вищої освіти,місто,Молодший спеціаліст,українська,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,443f89c2-84e7-4a56-bea3-005c776e0121,2004,жіноча,Кіровоградська область,м.Кропивницький,Фортечний район міста,Випускник загальноосвітнього навчального закла...,місто,Біолого-хімічний,українська,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0b7b2e81-a906-412f-ad74-005025fd9d29,2003,жіноча,Івано-Франківська область,Долинський район,м.Долина,Випускник загальноосвітнього навчального закла...,місто,Технологічний,українська,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# UCEQA / ZNO dataset field glossary

This dataset contains anonymized participant-level records from the Ukrainian Center for Educational Quality Assessment open data on external independent testing.
`NaN` usually means either "not applicable for this participant" or "no result/value recorded for this field" (for example, the participant did not register for that subject, did not attend, or the field only applies to current-year school graduates).

_Note: in 2021 "Ukraine literature and language" and "Ukraine language" were separate test categories. Before 2021, only "Ukraine literature and language" category existed._

## General participant fields

- `OUTID` — An anonymized unique participant identifier.
- `Birth` — Year of birth.
- `SexTypeName` — Sex of the participant.
- `RegName` — Registration region (oblast).
- `AREANAME` — Registration district / area.
- `TERNAME` — Registration settlement / city / locality.
- `RegTypeName` — Participant category, such as current-year school graduate or graduate of previous years.  
- `TerTypeName` — Type of settlement, such as city, urban-type settlement, or village.
- `ClassProfileNAME` — School class profile / specialization.
- `ClassLangName` — Language of instruction in the class or school.

## Educational institution fields

- `EONAME` — Name of the educational institution where the participant studies or studied.
- `EOTypeName` — Type of educational institution.
- `EORegName` — Region of the educational institution.
- `EOAreaName` — District / area of the educational institution.
- `EOTerName` — Settlement / locality of the educational institution.
- `EOParent` — Governing body or parent authority of the educational institution.

## Subject prefixes

- `UML` — Ukrainian language and literature.
- `Ukr` — Ukrainian language (2021).
- `Hist` — History of Ukraine.  
- `Math` — Mathematics.
- `MathSt` — Mathematics (standard level, 2021).  
- `Phys` — Physics.
- `Chem` — Chemistry.
- `Bio` — Biology.
- `Geo` — Geography. 
- `Eng` — English.  
- `Fra` — French.  
- `Deu` — German.  
- `Spa` — Spanish.  

## Generic subject field meanings

- `...Test` — Subject name / indicator that the participant registered for that subject.  
- `...SubTest` — Subject subtest indicator.  
- `...Lang` — Language of the test booklet / translation language.  
- `...TestStatus` — Test status, such as completed, absent, not passed threshold, or annulled.  
- `...Ball100` — Admission score on the 100–200 scale.  
- `...Ball12` — School assessment on the 1–12 scale used for DPA when applicable.  
- `...DPALevel` / `...DpaLevel` — DPA level, usually standard or profile level.  
- `...Ball` — Raw test score.  
- `...AdaptScale` — Adapted scoring scale flag, typically used for specific accommodation cases.  
- `...PTName` — Test center name.  
- `...PTRegName` — Test center region.  
- `...PTAreaName` — Test center district / area.  
- `...PTTerName` — Test center settlement / locality.  

## Some fields list

### General identity and registration

- `OUTID` — Anonymized unique participant ID.  
- `Birth` — Year of birth.  
- `SexTypeName` — Sex.  
- `RegName` — Registration region.  
- `AREANAME` — Registration area / district.  
- `TERNAME` — Registration locality.  
- `RegTypeName` — Registration / participant type.  
- `TerTypeName` — Locality type.  
- `ClassProfileNAME` — Class specialization profile.  
- `ClassLangName` — Language of instruction.  

### Educational institution

- `EONAME` — Educational institution name.  
- `EOTypeName` — Educational institution type.  
- `EORegName` — Educational institution region.  
- `EOAreaName` — Educational institution area / district.  
- `EOTerName` — Educational institution locality.  
- `EOParent` — Educational institution governing body.


In [8]:
all_column_sets = [set(df.columns) for df in years_dict.values()]
union_cols = set.union(*all_column_sets)
intersection_cols = set.intersection(*all_column_sets)
mismatched_cols = union_cols - intersection_cols

print(f"Total varying fields: {len(mismatched_cols)}")
print(mismatched_cols)

Total varying fields: 85
{'engball12', 'physball12', 'bioball', 'ukrball', 'spptareaname', 'chemball', 'engball', 'frtest', 'mathstpttername', 'umlball12', 'rusptregname', 'mathdpalevel', 'mathstlang', 'ruspttername', 'umlteststatus', 'fradpalevel', 'mathball', 'fraball12', 'frptregname', 'spptregname', 'umlptregname', 'frpttername', 'rusball12', 'fraptname', 'spptname', 'physball', 'rusptname', 'spateststatus', 'tertypename', 'geoball12', 'fraptareaname', 'classprofilename', 'spteststatus', 'mathstball', 'spaball12', 'bioball12', 'geoball', 'frateststatus', 'histball', 'stid', 'fraptregname', 'deudpalevel', 'frball100', 'classlangname', 'ukradaptscale', 'fratest', 'mathstball12', 'mathstptname', 'spaptname', 'frptname', 'frteststatus', 'umlpttername', 'rusball100', 'spapttername', 'spatest', 'chemball12', 'spadpalevel', 'frptareaname', 'fraball', 'spball100', 'mathstteststatus', 'fraball100', 'spaball100', 'ukrsubtest', 'sptest', 'umlptname', 'mathstptareaname', 'rusteststatus', 'umla

## Schema Discrepancies (Union minus Intersection)

By taking the union of all columns across all years and subtracting the intersection, we found exactly **57 fields** that are missing in at least one year.

We can categorize these 57 varying fields into six major structural changes:

### 1. Introduction of Raw Scores and DPA Grades
In earlier years (like 2016), the UCEQA primarily published the 100-200 point admission ratings. In later years, they expanded the datasets to include the raw test points (`...ball`) and the 1-12 scale school grades (`...ball12`) for the State Final Certification (DPA).
- **Raw Scores (`...ball`):** `mathball`, `engball`, `histball`, `physball`, `chemball`, `bioball`, `geoball`, `fraball`, `deuball`, `spaball`, `umlball`, `ukrball`.
- **DPA Scores (`...ball12`):** `umlball12`, `bioball12`, `chemball12`, `physball12`, `geoball12`, `engball12`, `fraball12`, `deuball12`, `spaball12`.

### 2. Discontinuation of the Russian Language Test
Russian language was offered as an optional subject in earlier years but was subsequently phased out of the external independent testing program.
- **Legacy `rus*` fields:** `rustest`, `rusteststatus`, `rusball100`, `rusball12`, `rusptname`, `rusptregname`, `rusptareaname`, `ruspttername`.

### 3. Separation of Ukrainian Language (2021)
Before 2021, participants took a combined "Ukrainian Language and Literature" exam. In 2021, the option to take only "Ukrainian Language" was introduced.
- **New independent fields:** `ukrsubtest`, `ukrball`, `ukradaptscale`.
- **Restructured combined fields:** `umltest`, `umlteststatus`, `umlball100`, `umlball`, `umladaptscale`, `umlptname`, `umlptregname`, `umlptareaname`, `umlpttername`. 

### 4. Mathematics Standard Level (2021)
When the DPA in mathematics became mandatory, a "Standard Level" track was created for students not pursuing STEM degrees. It provides a school grade but no 100-200 admission rating.
- **`mathst*` fields:** `mathsttest`, `mathstteststatus`, `mathstlang`, `mathstball`, `mathstball12`, `mathstptname`, `mathstptregname`, `mathstptareaname`, `mathstpttername`.

### 5. DPA Difficulty Levels (`...dpalevel`)
As high school curricula became more specialized, testing was adjusted so students could take exams at either a "Standard" or "Profile" level. Columns were added to indicate which level the student chose.
- **Level indicators:** `mathdpalevel`, `engdpalevel`, `fradpalevel`, `deudpalevel`, `spadpalevel`.

### 6. Enhanced School and Demographic Metadata
The UCEQA progressively added more detailed metadata to allow for deeper sociological and educational analysis, while phasing out older numerical identifiers.
- **New demographic fields:** `tertypename` (settlement type: city/village), `classprofilename` (class specialization), `classlangname` (language of instruction).
- **Legacy fields:** `stid` (an old numerical school identifier that was eventually replaced by comprehensive `eo*` text columns).


## Hypothesis 1: Regional Disparities in Educational Dynamics

**Hypothesis:** The quality of education over the observed period did not change uniformly across Ukraine. We hypothesize that macro-factors (such as educational reforms or the shift to remote learning) caused a disproportionate regional impact. Specifically, some regions will show a severe decline in average admission scores (a negative delta), while others will demonstrate resilience or improvement (a positive delta), widening the educational gap between different parts of the country.

We will calculate mean scores for different regions and differences (deltas) across years and create interactive map.

In [13]:
subjects = {
    'Math': 'mathball100',
    'English': 'engball100',
    'History': 'histball100',
    'Ukr. Lang & Lit': 'umlball100'
}
import re

def get_region_means_for_year(df, subjects_dict, year_label="Unknown"):
    all_cols = df.columns.tolist()
    
    found_mapping = {}
    
    search_patterns = {
        'Math': r'math.*ball100',
        'English': r'eng.*ball100',
        'History': r'hist.*ball100',
        'Ukr. Lang & Lit': r'(uml|ukr).*ball100'
    }
    
    for subj_name, pattern in search_patterns.items():
        match = [c for c in all_cols if re.search(pattern, c, re.IGNORECASE)]
        if match:
            found_mapping[subj_name] = match[0]
        else:
            print(f"Error: no subject found.")

    if not found_mapping:
        return pd.DataFrame()

    cols_to_use = ['regname'] + list(found_mapping.values())
    df_clean = df[cols_to_use].copy()
    
    for subj_name, real_col in found_mapping.items():
        df_clean[real_col] = (df_clean[real_col].astype(str)
                             .str.replace(r'\s+', '', regex=True)
                             .str.replace(',', '.', regex=False))
        
        df_clean[real_col] = pd.to_numeric(df_clean[real_col], errors='coerce')

    agg_df = df_clean.groupby('regname').mean()
    

    rename_map = {real_col: subjects_dict[subj_name] for subj_name, real_col in found_mapping.items()}
    agg_df = agg_df.rename(columns=rename_map)
    
    for target_col in subjects_dict.values():
        if target_col not in agg_df.columns:
            agg_df[target_col] = np.nan
            
    return agg_df


years_dict = {int(k): v for k, v in years_dict.items()}
available_years = sorted(list(years_dict.keys()))

processed_data = {}
for yr in available_years:
    processed_data[yr] = get_region_means_for_year(years_dict[yr], subjects, year_label=yr)

In [14]:
gdf_base = gpd.read_file(next(data_path.glob("*.geojson")))

In [17]:
def update_map_range(year_range, subject_label):
    start_year, end_year = year_range
    subj_col = subjects[subject_label]
    
    df_start = processed_data[start_year]
    df_end = processed_data[end_year]
    
    delta_series = (df_end[subj_col] - df_start[subj_col]).reset_index()
    
    delta_series['regname'] = delta_series['regname'].replace({'м.Київ': 'Київ'})
    
    gdf_plot = gdf_base.merge(delta_series, left_on='name:ua', right_on='regname', how='left')
    
    fig, ax = plt.subplots(figsize=(12, 7), facecolor='#111111')
    ax.set_facecolor('#111111')
    
    cmap = plt.get_cmap('RdBu')
    norm = mcolors.Normalize(vmin=-30, vmax=30)
    
    gdf_base.plot(ax=ax, color='#222222', edgecolor='#444444', linewidth=0.5)
    
    gdf_plot.plot(
        column=subj_col,
        ax=ax,
        cmap=cmap,
        norm=norm,
        edgecolor='black',
        linewidth=0.6,
        missing_kwds={'color': '#222222'}
    )
    
    # Add Colorbar
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    cbar = fig.colorbar(sm, ax=ax, fraction=0.02, pad=0.02)
    cbar.ax.yaxis.set_tick_params(color='white', labelcolor='white')
    cbar.set_label('Score Delta', color='white')
    
    title_text = f"{subject_label} Change: {end_year} vs {start_year}"
    if start_year == end_year:
        title_text = f"{subject_label} Snapshot: {start_year} (Delta is 0)"
        
    ax.set_title(title_text, color='white', fontsize=14, pad=20)
    ax.axis('off')
    plt.show()

range_slider = IntRangeSlider(
    value=[available_years[0], available_years[-1]],
    min=min(available_years),
    max=max(available_years),
    step=1,
    description='Years Range:',
    continuous_update=False,
    style={'description_width': 'initial'},
    layout={'width': '500px'}
)

subject_dropdown = Dropdown(
    options=list(subjects.keys()),
    value=list(subjects.keys())[0],
    description='Subject:',
    style={'description_width': 'initial'}
)

interact(update_map_range, year_range=range_slider, subject_label=subject_dropdown);

interactive(children=(IntRangeSlider(value=(2016, 2021), continuous_update=False, description='Years Range:', …

### Analysis of Hypothesis 1

**Conclusion: Partially Supported (Subject-Dependent)**

The data partially supports the hypothesis. While regional disparities are indeed visible, the most striking finding is that the impact is overwhelmingly **subject-dependent** rather than purely regional. Macro-factors (such as the shift to remote learning between 2018 and 2021) affected certain disciplines uniformly, while regional disparities manifested strongly in others.

#### Breakdown by Subject:

- **Math (The Universal Decline)** -- The map is entirely red, indicating a negative score delta across the entire country. This contradicts the hypothesis that *some* regions would demonstrate resilience while others declined. The decline was a macro-trend affecting all regions, likely highlighting the universal difficulty of adapting STEM subjects to remote learning. However, there is disparity in the severity of the decline, with central, southern, and western regions experiencing a much steeper drop.
- **History (The Clear Regional Split)** -- The map displays a mix of blue (improvement) and red/orange (decline). The far western regions show positive deltas, while the central, southern, and eastern regions show negative deltas. This strongly supports the hypothesis, showing a disproportionate regional impact that widens the educational gap in this specific subject.
- **Ukrainian Language & Literature (The Resilient Outliers)** -- This map is predominantly blue, indicating general nationwide improvement or stability. While the macro-trend is positive, regional disparities are still visible. The far southwestern regions show a massive improvement compared to the rest of the country. This supports the idea of widening regional gaps, but due to some regions accelerating much faster than the national average.
- **English (The Uniformly Stable)** -- The map is dominated by very pale blues and whites, showing near-zero to slight positive changes almost everywhere. This does not support the hypothesis of widening regional disparities, as educational dynamics for English remained relatively uniform and stable across the country.

#### Summary
The hypothesis correctly anticipates that macro-factors caused uneven impacts, but the maps reveal a critical nuance: vulnerability to systemic changes is heavily dictated by the subject matter. Math suffered universally, English remained stable universally, while History and Ukrainian Language & Literature exhibited the severe regional disparities the hypothesis predicted.

In [20]:
print("-" * 70)

----------------------------------------------------------------------


## Hypothesis 2: Educational Inequality Index (Variance and Elite Concentration)

**Hypothesis:** A high average score in a region can be misleading. In some regions, a high average is achieved through uniformly good preparation across all schools (low variance/standard deviation). In others, it is driven by a few elite specialized lyceums located in the regional center, while rural or general schools perform poorly, creating a polarized distribution (high variance). 

To test this, we will look at the **Standard Deviation** of scores and the **Concentration of Elite Students** (percentage of participants scoring $\ge 195$) for Mathematics and English in the latest available year. 
- **High Standard Deviation** indicates severe educational inequality and a large gap between the best and worst students.
- **High Elite Percentage** indicates a strong concentration of top-tier talent or specialized schools.


In [24]:
target_year = max(available_years)
print(f"Analyzing inequality for the year {target_year}")

df_target = years_dict[target_year].copy()

df_target['math_clean'] = np.nan
df_target['eng_clean'] = np.nan

def get_col(pattern):
    matches = [c for c in df_target.columns if re.search(pattern, c, re.IGNORECASE)]
    return matches[0] if matches else None

math_col = get_col(r'math.*ball100')
eng_col = get_col(r'eng.*ball100')

for raw_col, clean_col in [(math_col, 'math_clean'), (eng_col, 'eng_clean')]:
    if raw_col:
        df_target[clean_col] = (df_target[raw_col].astype(str)
                                .str.replace(r'\s+', '', regex=True)
                                .str.replace(',', '.', regex=False))
        df_target[clean_col] = pd.to_numeric(df_target[clean_col], errors='coerce')

def calc_inequality(series):
    s = series.dropna()
    if len(s) == 0:
        return pd.Series({'std': np.nan, 'elite_pct': np.nan})
    std_val = s.std()
    elite_pct = (s[s >= 195].count() / len(s)) * 100
    return pd.Series({'std': std_val, 'elite_pct': elite_pct})

ineq_stats = df_target.groupby('regname').apply(
    lambda x: pd.Series({
        'Math_STD': calc_inequality(x['math_clean'])['std'],
        'Math_Elite': calc_inequality(x['math_clean'])['elite_pct'],
        'Eng_STD': calc_inequality(x['eng_clean'])['std'],
        'Eng_Elite': calc_inequality(x['eng_clean'])['elite_pct']
    })
).reset_index()

ineq_stats['regname'] = ineq_stats['regname'].replace({'м.Київ': 'Київ'})

print("Top 3 Regions with Highest Math Inequality (Standard Deviation):")
display(ineq_stats[['regname', 'Math_STD', 'Math_Elite']].sort_values(by='Math_STD', ascending=False).head(3))


Analyzing inequality for the year 2021
Top 3 Regions with Highest Math Inequality (Standard Deviation):


,regname,Math_STD,Math_Elite
23,Чернігівська область,68.690755,0.542050
6,Закарпатська область,68.533580,0.419036
9,Кіровоградська область,68.002456,0.362611


In [25]:
def plot_inequality_map(metric, scale_mode):
    # Determine absolute min/max vs relative min/max
    actual_min = ineq_stats[metric].min()
    actual_max = ineq_stats[metric].max()
    
    if 'STD' in metric:
        title = f"Score Variance (Standard Deviation) - {target_year}"
        cmap = 'magma'
        cbar_label = 'Standard Deviation'
        vmin = 40 if scale_mode == 'Strict' else actual_min
        vmax = 70 if scale_mode == 'Strict' else actual_max
    else:
        title = f"Elite Concentration (% scoring ≥ 195) - {target_year}"
        cmap = 'inferno'
        cbar_label = '% of Valid Participants'
        vmin = 0
        vmax = 5.0 if scale_mode == 'Strict' else actual_max

    gdf_ineq = gdf_base.merge(ineq_stats, left_on='name:ua', right_on='regname', how='left')
    
    fig, ax = plt.subplots(figsize=(12, 7), facecolor='#111111')
    ax.set_facecolor('#111111')
    
    gdf_base.plot(ax=ax, color='#222222', edgecolor='#444444', linewidth=0.5)
    
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    gdf_ineq.plot(
        column=metric,
        ax=ax,
        cmap=cmap,
        norm=norm,
        edgecolor='black',
        linewidth=0.6,
        missing_kwds={'color': '#222222'}
    )
    
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    cbar = fig.colorbar(sm, ax=ax, fraction=0.02, pad=0.02)
    cbar.ax.yaxis.set_tick_params(color='white', labelcolor='white')
    cbar.set_label(cbar_label, color='white', labelpad=10)
    
    mode_text = f"[{scale_mode} Scale: {vmin:.1f} to {vmax:.1f}]"
    ax.set_title(f"{title}\n{mode_text}", color='white', fontsize=14, pad=20)
    ax.axis('off')
    
    spread_text = f"Actual Data Spread: {actual_min:.1f} to {actual_max:.1f}"
    ax.text(0.5, -0.05, spread_text, transform=ax.transAxes, 
            color='#aaaaaa', ha='center', fontsize=10)
    
    plt.show()

metric_dropdown = Dropdown(
    options=[
        ('Mathematics (Std Dev)', 'Math_STD'),
        ('English (Std Dev)', 'Eng_STD'),
        ('Mathematics (Elite %)', 'Math_Elite'),
        ('English (Elite %)', 'Eng_Elite')
    ],
    value='Math_STD',
    description='Metric:'
)

scale_dropdown = Dropdown(
    options=['Relative', 'Strict'],
    value='Relative',
    description='Scale Mode:',
    style={'description_width': 'initial'}
)

interact(plot_inequality_map, metric=metric_dropdown, scale_mode=scale_dropdown);


interactive(children=(Dropdown(description='Metric:', options=(('Mathematics (Std Dev)', 'Math_STD'), ('Englis…

### Analysis of Hypothesis 2: Educational Inequality Index

**Conclusion: Partially Supported (Metric-Dependent)**

The data partially supports the hypothesis. While we hypothesized that regions would exhibit different levels of overall internal inequality (variance), the maps reveal that the general spread of scores is a national constant. However, severe regional disparities become blatantly obvious when we isolate the extreme upper end of the distribution (the elite performers). 

#### Breakdown by Metric:

- **Standard Deviation (The Systemic Constant)** -- While the relative color map initially suggested regional contrasts, strict numerical scaling reveals that the actual variance is remarkably narrow (e.g., fluctuating only between 65.5 and 68.5 for Math across the entire country). This *contradicts* the hypothesis that some regions are highly equitable while others are highly polarized. The "gap between the best and the worst" is a systemic, nationwide constant. This implies that the true divide (likely urban vs. rural) exists identically within every region, rather than differentiating the regions from one another.
- **Elite Concentration (The Capital Monopoly)** -- When measuring the percentage of participants scoring $\ge 195$, the map shows massive, localized positive outliers, primarily Kyiv City and a few other major metropolitan hubs. This strongly supports the second half of the hypothesis. The data confirms an "elite clustering" effect: regions with highly specialized physical-mathematical or linguistic lyceums disproportionately hoard top-tier talent, skewing their averages upward without actually raising the floor for the majority of their students.
- **Subject Differences (Math vs. English)** -- Although geographically uniform, the absolute level of inequality depends heavily on the subject. Math exhibits a very high standard deviation (~67 points) nationwide, indicating a fundamentally polarized subject where students either excel or completely fail. English shows a noticeably tighter spread (~52 points), suggesting slightly more consistent baseline outcomes across the student body, even if the elite scores remain geographically clustered.

#### Summary
The hypothesis correctly anticipates that regional averages are artificially inflated by elite clustering, but it incorrectly assumes that overall variance fluctuates by geography. The maps reveal a critical nuance: the "shape" of the educational bell curve (the overall variance) is nearly identical in every single oblast. True geographical disparity does not exist in the width of the curve, but strictly in the extreme right tail, which is heavily monopolized by the capital.
